# Attention: от Q/K/V до causal mask

**Attention Is All You Need** · Vaswani et al. · 2017

[Оригинальная статья](https://arxiv.org/abs/1706.03762)

**Цель урока:** Реализовать scaled dot-product attention без готового attention-модуля и проверить, что будущее не меняет прошлое.

## Перед началом

Базовые тензоры PyTorch, матричное умножение, softmax и backward. Сначала пройдите «Как модель учится».

Около 40–60 минут с заданиями. CPU, PyTorch и Matplotlib; без скачивания весов и датасетов. Время — ориентир для чтения и работы, а не benchmark выполнения.

## Задача и идея

До Transformer обмен информацией в последовательности часто зависел от рекуррентного прохода. Attention позволяет каждой позиции собрать взвешенную смесь признаков других позиций. Q описывает запрос позиции, K — признаки для сопоставления, V — передаваемую информацию. Это обучаемые проекции, а не заранее заданные роли слов.

Вектор результата — смесь V, но веса смеси зависят от Q и K. Несколько голов работают в разных подпространствах. Сам по себе attention не знает порядок: нужны позиционные признаки. В causal LM отдельно запрещают доступ к будущим позициям.

## Граница воспроизведения

Только attention-механизм. Нет полного encoder–decoder, positional encoding, translation corpus и воспроизведения BLEU из статьи.

Код написан для этого урока по математическим идеям статьи. Это не скопированный авторский репозиторий. Успех assertions подтверждает указанные свойства реализации, но не результаты статьи.

## Формула → тензоры

`S = Q @ K.transpose(-2,-1) / sqrt(d_head)`; `P = softmax(S + mask)`; `O = P @ V`.

При форме Q/K/V `[B,H,T,D]` scores имеют форму `[B,H,T,T]`. Softmax идёт по ключам (последняя ось); сумма каждой строки равна 1. Масштаб уменьшает рост дисперсии dot product. В mask разрешённые элементы равны 0, запрещённые — `-inf`. Полностью запрещённая строка даст NaN, поэтому диагональ оставляем доступной.

In [ ]:
%matplotlib inline
import math
import torch
from torch import nn
from torch.nn import functional as F
import matplotlib.pyplot as plt
torch.set_num_threads(1)
torch.manual_seed(42)
plt.rcParams.update({'figure.figsize': (7, 3), 'axes.spines.top': False, 'axes.spines.right': False})
print('PyTorch:', torch.__version__, '| CPU | seed=42')

## Шаг 1

Явно считаем scores и веса. Посмотрите на causal треугольник: верхняя часть должна быть нулевой после softmax.

In [ ]:
def attention(q, k, v, causal=True):
    scores = q @ k.transpose(-2, -1) / q.shape[-1]**0.5
    if causal:
        future = torch.triu(torch.ones(q.shape[-2], k.shape[-2], dtype=torch.bool), diagonal=1)
        scores = scores.masked_fill(future, float('-inf'))
    probabilities = scores.softmax(dim=-1)
    return probabilities @ v, probabilities
q, k, v = [torch.randn(2, 2, 6, 4) for _ in range(3)]
out, probabilities = attention(q, k, v)
assert out.shape == q.shape
assert torch.allclose(probabilities.sum(-1), torch.ones(2, 2, 6))
assert torch.equal(probabilities.triu(1), torch.zeros_like(probabilities))
plt.imshow(probabilities[0, 0].detach(), cmap='Reds', vmin=0, vmax=1)
plt.xlabel('key position'); plt.ylabel('query position'); plt.colorbar(); plt.show()

## Шаг 2

Одна голова становится модулем с обучаемыми Q/K/V. Градиент должен доходить до всех трёх проекций.

In [ ]:
class AttentionBlock(nn.Module):
    def __init__(self, width=8):
        super().__init__()
        self.q = nn.Linear(width, width, bias=False)
        self.k = nn.Linear(width, width, bias=False)
        self.v = nn.Linear(width, width, bias=False)
    def forward(self, x):
        return attention(self.q(x), self.k(x), self.v(x))[0]
block = AttentionBlock(); x = torch.randn(2, 6, 8)
target = torch.randn_like(x)
F.mse_loss(block(x), target).backward()
for name, parameter in block.named_parameters():
    print(name, 'gradient norm:', parameter.grad.norm().item())
    assert torch.isfinite(parameter.grad).all() and parameter.grad.norm() > 0

## Шаг 3

Меняем только будущую часть входа. Причинность проверяется выходами, а не названием класса.

In [ ]:
changed = x.clone(); changed[:, 3:] += 10 * torch.randn_like(changed[:, 3:])
with torch.no_grad():
    delta = (block(x)[:, :3] - block(changed)[:, :3]).abs().max()
    unmasked_a = attention(block.q(x), block.k(x), block.v(x), False)[0]
    unmasked_b = attention(block.q(changed), block.k(changed), block.v(changed), False)[0]
print('causal delta:', delta.item(), 'unmasked delta:', (unmasked_a[:, :3] - unmasked_b[:, :3]).abs().max().item())
assert delta < 1e-6

## Как читать результат

Нулевая верхняя часть attention matrix и неизменность раннего выхода — две независимые проверки. Ненулевой градиент показывает наличие пути обучения, но не качество language modeling. Heatmap относится к случайным входам: нельзя приписывать ей языковой смысл.

## Самостоятельные эксперименты

1. Уберите sqrt(D) и сравните entropy весов при D=4 и D=256.
2. Уберите mask: какой assert должен упасть?
3. Добавьте две головы и выходную Linear, сохранив ширину 8.

Сначала запишите прогноз, затем меняйте по одному фактору. Для честного сравнения сбрасывайте ядро и выполняйте все ячейки с начала. Сохраните копию для своих изменений.

## Проверка рассуждений — после эксперимента

Для нормальных независимых Q/K variance dot product растёт с D, поэтому без масштаба softmax часто становится резче. Без mask ранний output зависит от будущего. Две головы: `[B,T,8] → [B,2,T,4]`, затем concatenate и проекция 8→8.

## Что записать в лабораторный журнал

- Гипотеза и изменённый параметр.
- Seed, размер данных, число шагов и используемая precision.
- Значения проверок до и после, график и объяснение расхождения с прогнозом.
- Что этот эксперимент **не** позволяет заключить.

Связанный раздел доступен по кнопке «Открыть раздел студии» над уроком.